In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import statsmodels.api as sm

In [3]:
# List of tickers to use in the analysis
tickers = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'AGG', 'LQD', 'HYG', 'EMB', 'GLD', 'SLV',
           'VNQ', 'VWO', 'VEA', 'TIP', 'XLF', 'XLK', 'XLE', 'XLY', 'XLP', 'XLI', 'XLB',
           'XLU', 'XLC', 'XBI', 'EWJ', 'EWZ', 'EWA', 'FXI', 'GDX', 'TLT']

# Loading data
prices_df =  yf.download(tickers, start='2024-01-01', end='2025-01-06', auto_adjust=True)['Close']

# Remove columns (stocks) with more than 10% missing values
prices_df = prices_df.dropna(axis=1, thresh=len(prices_df) * 0.9)

# Fill any remaining missing values with forward fill then backward fill
prices_df = prices_df.ffill().bfill()

[*********************100%***********************]  31 of 31 completed


In [23]:
# As our method takes in returns, we'll calculate them from our dataframe of prices
returns_df = prices_df.pct_change().dropna()

In [6]:
t = 252
current_date = returns_df.index[t]

In [7]:
# Get training data (PCA window)
pca_window = 252
train_returns = returns_df.iloc[t-pca_window:t]

In [8]:
# Perform PCA
# Standardize returns
n_components = None
standardized_returns = (train_returns - train_returns.mean()) / train_returns.std()
pca = PCA(n_components=n_components)
pca.fit(standardized_returns)

PCA()

In [9]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
K = np.argmax(cumulative_variance >= 0.85) + 1

In [11]:
# Get last ou_window days for OU process fitting
ou_window = 60
ou_estimation_returns = train_returns.iloc[-ou_window:]

In [24]:
# Calculate factor returns
factor_weights = pd.DataFrame(pca.components_, columns=train_returns.columns) / train_returns.std()

factor_returns = pd.DataFrame(
        np.dot(ou_estimation_returns, factor_weights.transpose()),
        index=ou_estimation_returns.index
    )

In [17]:
# Fit regression for each stock to get residuals
residuals = pd.DataFrame(index=ou_estimation_returns.index, columns=ou_estimation_returns.columns)
factor_models = {}

# Fit regression for each stock
for stock in ou_estimation_returns.columns:
    # Get stock returns
    y = ou_estimation_returns[stock]
    
    # Add constant to factor returns for intercept
    X = factor_returns.iloc[:, :K]  # Use only first K factors
    X = sm.add_constant(X)
    
    # Fit regression model
    model = sm.OLS(y, X).fit()
    factor_models[stock] = model
    residuals[stock] = model.resid

In [25]:
# Initialize results dictionary
ou_params = {}
tradable_stocks = []

kappa_threshold = 8.4

# Process each asset
for stock in residuals.columns:
    # Fit discrete Ornstein-Uhlenbeck process to residual time series
    y = residuals[stock].cumsum().values[1:]
    x = residuals[stock].cumsum().values[:-1]
    n = len(y)
    
    valid = False

    # Add constant to x for intercept
    X = sm.add_constant(x)
    
    # Fit OLS model
    model = sm.OLS(y, X).fit()
    
    # Extract parameters
    a, b = model.params  # a is intercept, b is slope

    # Check validity
    if 0 < b < 1 and model.pvalues[1] < 0.05:
        valid = True
    
    if valid:
        # Calculate OU parameters
        kappa = -np.log(b) * 252 # Mean reversion speed
        m = a / (1 - b)  # Long-term mean
        sigma_eq = np.sqrt(np.sum(model.resid**2) / (n - 1) / (1 - b**2))
    else:
        kappa, m, sigma_eq = None, None, None
    
    if kappa is not None and kappa > kappa_threshold:
        ou_params[stock] = {'kappa': kappa, 'm': m, 'sigma_eq': sigma_eq}
        tradable_stocks.append(stock)

In [33]:
# Demean m across tradable stocks
m_vec = np.array([ou_params[stock]['m'] for stock in tradable_stocks])
m_mean = np.mean(m_vec)
for stock in tradable_stocks:
    ou_params[stock]['m'] -= m_mean

# Precompute all S-scores in a dictionary
s_score_data = {}
for stock in tradable_stocks:
    m = ou_params[stock]['m']
    sigma_eq = ou_params[stock]['sigma_eq']
    s_score = (residuals[stock].cumsum() - m) / sigma_eq

    s_score_data[stock] = s_score

s_scores = pd.DataFrame(s_score_data, index=residuals.index)

In [39]:
# Generate signals for the last day
last_day_scores = s_scores.iloc[-1:]

signals = pd.DataFrame(0, index=last_day_scores.index, columns=last_day_scores.columns)

# Long signal when S-score is below negative threshold
signals[s_scores < -1.25] = 1

# Short signal when S-score is above positive threshold
signals[s_scores > 1.25] = -1

In [83]:
beta_matrix = pd.DataFrame({
    stock: model.params.drop('const') 
    for stock, model in factor_models.items()
}).T

In [85]:
portfolio_positions = pd.DataFrame(0.0, index=returns_df.index[pca_window:], columns=returns_df.columns)

for stock in tradable_stocks:
    signal = signals.iloc[0][stock]

    if signal != 0:
        # 1. Long $1 of the stock
        portfolio_positions.loc[current_date, stock] += signal * 1

        # 2. Get betas for first K factors
        stock_betas = beta_matrix.loc[stock, :].values

        # 3. Get factor weights matrix (factors x assets)
        factor_weights_subset = factor_weights.values[:K, :]  # Shape: (K, num_assets)

        # 4. Calculate hedge positions: -signal * (betas @ factor_weights)
        hedge_positions = -signal * (stock_betas @ factor_weights_subset)  # Shape: (num_assets,)

        # 5. Apply positions to all assets
        portfolio_positions.loc[current_date] += hedge_positions

In [ ]:
strategy_returns = pd.Series(index=returns_df.index[252:], dtype=float)
if t+1 <= len(returns_df):
    next_day_returns = returns_df.iloc[t]
    day_return = 0
    
    for stock in tradable_stocks:
        position = portfolio_positions.loc[current_date, stock]
        if not np.isnan(position) and position != 0:
            day_return += position * next_day_returns[stock]
    
    strategy_returns.loc[current_date] = day_return